In [1]:
   import json
   import time

   import pandas as pd
   import requests

   from analytics.db import query
   from analytics.paths import RAW_WEB
   from analytics.web import get

In [2]:
query("""
      --w09
    SELECT 
    COUNT(*) FILTER (WHERE product_weight_g > 10000) AS heavy_products_count,
    COUNT(*) AS total_known_weight,
    ROUND(CAST(100.0 * COUNT(*) FILTER (WHERE product_weight_g > 10000) / COUNT(*) AS NUMERIC), 1) AS percentage
    FROM products
    WHERE product_weight_g IS NOT NULL;
      """)

,heavy_products_count,total_known_weight,percentage
0,1891,32949,5.7


**Explanation:** This query counts products with a weight greater than 10,000 grams (10 kg) and calculates their percentage relative to all products that have a non-null (known) weight, rounded to one decimal place.

## Part 1 — Ask with Python, and save the answer

In [3]:
response = requests.get("https://brasilapi.com.br/api/feriados/v1/2017", timeout=30)
response.status_code

200

Every answer has a **status code**: a number that says how the request went. `200` means OK. `timeout=30` means: wait at most 30 seconds for the answer. Without a timeout, a request can wait forever.

In [4]:
holidays_2017 = response.json()
len(holidays_2017), holidays_2017[0]

(13,
 {'date': '2017-01-01',
  'name': 'Confraternização mundial',
  'type': 'national',
  'weekday': 'domingo'})

This cell parses the JSON response into Python lists and dictionaries, returning the total count of holidays and inspecting the first record.

In [5]:
   path = RAW_WEB / "brasilapi" / "holidays-2017.json"
   path.parent.mkdir(parents=True, exist_ok=True)
   path.write_text(response.text, encoding="utf-8")

   json.loads(path.read_text(encoding="utf-8")) == holidays_2017

True

This cell saves the raw API response text directly into a local JSON file, ensuring data persistence and verifying that reloading the file yields identical data structures.

In [6]:
   for year in [2016, 2017, 2018]:
       path = RAW_WEB / "brasilapi" / f"holidays-{year}.json"
       if path.exists():
           continue
       response = requests.get(f"https://brasilapi.com.br/api/feriados/v1/{year}", timeout=30)
       response.raise_for_status()
       path.write_text(response.text, encoding="utf-8")
       time.sleep(1)

   sorted(p.name for p in (RAW_WEB / "brasilapi").glob("*.json"))

['holidays-2016.json', 'holidays-2017.json', 'holidays-2018.json']

This loop checks and downloads missing holiday JSON files for 2016, 2017, and 2018 sequentially with a polite one-second delay between requests, then lists all saved files.

In [7]:
   rows = []
   for path in sorted((RAW_WEB / "brasilapi").glob("holidays-*.json")):
       rows.extend(json.loads(path.read_text(encoding="utf-8")))

   holidays = pd.DataFrame(rows)
   holidays["date"] = pd.to_datetime(holidays["date"], format="%Y-%m-%d")
   holidays.shape

(39, 4)

This cell aggregates all locally saved holiday JSON files into a single pandas DataFrame, formats the date column, and displays the final shape of the dataset.

In [8]:
#1.a
weekend_holidays = holidays["date"].dt.dayofweek.isin([5, 6]).sum()
weekend_holidays

np.int64(7)

This cell calculates how many holidays fall on a weekend (Saturday [5] or Sunday [6]) by checking the day of the week for each date.

In [9]:
#1.b
variable_holidays = (
    holidays.assign(month_day=holidays["date"].dt.strftime("%m-%d"))
    .groupby("name")["month_day"]
    .nunique()
    .loc[lambda x: x > 1]
)
variable_holidays


name
Carnaval             6
Corpus Christi       3
Páscoa               3
Sexta-feira Santa    3
Name: month_day, dtype: int64

This cell identifies holidays that have shifting dates across different years by grouping holiday names and counting how many unique month-day values they possess.

## Part 2 — Nested JSON, one page at a time

In [10]:
   PTAX_URL = (
       "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
   )
   params = {
       "@dataInicial": "'09-01-2016'",
       "@dataFinalCotacao": "'10-31-2018'",
       "$format": "json",
       "$orderby": "dataHoraCotacao",
       "$top": 100,
       "$skip": 0,
   }
   response = requests.get(PTAX_URL, params=params, timeout=30)
   page = response.json()
   page.keys()

dict_keys(['@odata.context', 'value'])

This cell defines the PTAX API endpoint and its query parameters (date range, format, sorting, and pagination limits) to fetch the first page of daily USD exchange rates from the Central Bank of Brazil.

In [11]:
 page["value"][:2]

[{'cotacaoCompra': 3.2466,
  'cotacaoVenda': 3.2472,
  'dataHoraCotacao': '2016-09-01 13:10:10.989'},
 {'cotacaoCompra': 3.2425,
  'cotacaoVenda': 3.2431,
  'dataHoraCotacao': '2016-09-02 13:05:51.688'}]

In [12]:
pd.json_normalize(page, record_path="value").shape

(100, 3)

This cell inspects the nested structure of the API response and flattens the list of exchange rate records found under the `value` key into a tabular pandas DataFrame.

In [13]:
   skip = 0
   while True:
       path = RAW_WEB / "bcb" / f"ptax-{skip:04d}.json"
       if not path.exists():
           response = requests.get(PTAX_URL, params=params | {"$skip": skip}, timeout=30)
           response.raise_for_status()
           path.parent.mkdir(parents=True, exist_ok=True)
           path.write_text(response.text, encoding="utf-8")
           time.sleep(1)
       rows = json.loads(path.read_text(encoding="utf-8"))["value"]
       print(path.name, len(rows))
       if len(rows) < 100:
           break
       skip += 100

ptax-0000.json 100
ptax-0100.json 100
ptax-0200.json 100
ptax-0300.json 100
ptax-0400.json 100
ptax-0500.json 42


This loop paginates through the PTAX API by incrementing the `$skip` parameter, saving each page locally as a separate JSON file until a page returns fewer than 100 rows.

In [14]:
   pages = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((RAW_WEB / "bcb").glob("ptax-*.json"))]
   rates = pd.concat([pd.json_normalize(page, record_path="value") for page in pages], ignore_index=True)
   rates = pd.DataFrame({
       "date": pd.to_datetime(rates["dataHoraCotacao"].str[:10], format="%Y-%m-%d"),
       "buy": rates["cotacaoCompra"],
       "sell": rates["cotacaoVenda"],
   })
   rates.shape, rates["date"].is_unique

((542, 3), True)

This cell aggregates all locally saved PTAX pages, normalizes the nested JSON structure, formats the date and exchange rate columns, and validates that each date has a unique exchange rate.

In [15]:
#2.a
max_row = rates.loc[rates["sell"].idxmax()]
min_row = rates.loc[rates["sell"].idxmin()]
max_row, min_row

(date    2018-09-14 00:00:00
 buy                  4.1873
 sell                 4.1879
 Name: 509, dtype: object,
 date    2017-02-16 00:00:00
 buy                  3.0504
 sell                  3.051
 Name: 116, dtype: object)

This cell finds the exact dates and prices when the US dollar reached its maximum and minimum selling values during the analyzed period.

In [16]:
#2.b
diff = rates["sell"] - rates["buy"]
max_diff = diff.max()
max_diff_pct = (max_diff / rates.loc[diff.idxmax(), "sell"]) * 100
max_diff, max_diff_pct

(np.float64(0.0051000000000001044), np.float64(0.15085633152897637))

This cell calculates the maximum absolute spread between the selling and buying rates, expresses it as a percentage of the selling price, and evaluates its impact on business reporting.

## Part 3 — When a request fails

In [17]:
response = requests.get("https://brasilapi.com.br/api/feriados/v1/1800", timeout=30)
response.status_code, response.json()["message"]

(404, 'Ano fora do intervalo suportado entre 1900 e 2199.')

This cell tests an invalid API request for a non-existent year (1800) and demonstrates how `raise_for_status()` catches HTTP errors.

In [18]:
   try:
       response.raise_for_status()
   except requests.HTTPError as error:
       print(error)

404 Client Error: Not Found for url: https://brasilapi.com.br/api/feriados/v1/1800


This cell simulates a network timeout by setting an extremely short timeout threshold, illustrating how connection failures are intercepted.

In [19]:
try:
       requests.get("https://brasilapi.com.br/api/feriados/v1/2017", timeout=0.001)
except requests.Timeout as error:
       print(type(error).__name__)

ReadTimeout


In [20]:
get("https://brasilapi.com.br/api/feriados/v1/2017").status_code

200

In [21]:
   try:
       get("https://brasilapi.com.br/api/feriados/v1/1800")
   except requests.HTTPError as error:
       print(error)

   try:
       get("https://brasilapi.com.br/api/feriados/v1/2017", timeout=0.001)
   except RuntimeError as error:
       print(error)

404 Client Error: Not Found for url: https://brasilapi.com.br/api/feriados/v1/1800
Try 1 of 3 failed: ReadTimeout
Try 2 of 3 failed: ReadTimeout
Try 3 of 3 failed: ReadTimeout
https://brasilapi.com.br/api/feriados/v1/2017 failed 3 times. The last problem: ReadTimeout


This cell tests the custom robust `get` function from `web.py`, verifying that it handles successful requests, immediately aborts on 404 errors without retrying, and retries on timeouts before raising a RuntimeError.

In [22]:
#3.a
try:
    get(PTAX_URL, params={"@dataInicial": "'31-12-2017'", "@dataFinalCotacao": "'10-31-2018'", "$format": "json"})
except Exception as error:
    print(type(error).__name__, error)

Try 1 of 3 failed: HTTP 500
Try 2 of 3 failed: HTTP 500
Try 3 of 3 failed: HTTP 500
RuntimeError https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao) failed 3 times. The last problem: HTTP 500


This cell tests the PTAX API with an inverted date format (`31-12-2017`), demonstrating that it fails because the API expects the month-day-year format.

**3.b**: get does not retry after a 404 because a 404 status code means the requested resource does not exist. Repeating an identical request for a non-existent resource will always fail, so retrying is pointless.

## Part 4 — From saved files to clean tables

In [23]:
   rates = query("SELECT * FROM clean.usd_rates ORDER BY date")
   holidays = query("SELECT * FROM clean.holidays ORDER BY date")

   days = pd.DataFrame({"date": pd.date_range("2016-09-01", "2018-10-31")})
   days["weekend"] = days["date"].dt.dayofweek >= 5
   days["has_price"] = days["date"].isin(rates["date"])
   days["holiday"] = days["date"].isin(holidays["date"])
   days.groupby(["weekend", "has_price", "holiday"]).size().reset_index(name="days")

,weekend,has_price,holiday,days
0,False,False,True,23
1,False,True,False,542
2,True,False,False,221
3,True,False,True,5


This cell cross-references the holidays and USD exchange rate tables against a complete daily date range to verify data consistency and check missing rates against holidays.

Ex a.4 in schemas.py & wep.py files 

In [24]:
#4.b
missing_price_days = days[(days["weekend"] == False) & (days["has_price"] == False) & (days["holiday"] == True)]
missing_price_days.merge(holidays, on="date", how="left")

,date,weekend,has_price,holiday,name
0,2016-09-07,False,False,True,Independência do Brasil
1,2016-10-12,False,False,True,Nossa Senhora Aparecida
2,2016-11-02,False,False,True,Finados
3,2016-11-15,False,False,True,Proclamação da República
4,2017-02-27,False,False,True,Carnaval
5,2017-02-28,False,False,True,Carnaval
6,2017-04-14,False,False,True,Sexta-feira Santa
7,2017-04-21,False,False,True,Tiradentes
8,2017-05-01,False,False,True,Dia do trabalho
9,2017-06-15,False,False,True,Corpus Christi


This cell lists the 23 weekdays lacking exchange rates, confirming they correspond precisely to national holidays such as Carnaval.

In [25]:
sales = query("""
    SELECT date_trunc('day', o.order_purchase_timestamp) AS date,
           SUM(i.price + i.freight_value) AS sales_brl
    FROM clean.order_items AS i
    JOIN clean.orders AS o ON o.order_id = i.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY 1
    ORDER BY 1
""")
sales.shape, round(sales["sales_brl"].sum(), 2)


((602, 2), np.float64(15683706.74))

This cell queries daily total sales in Brazilian Reais (BRL) from clean orders and items, filtering out canceled and unavailable orders between January 2017 and August 2018.

In [26]:
inner = sales.merge(rates, on="date", how="inner", validate="one_to_one")
len(inner), round(inner["sales_brl"].sum(), 2)

(413, np.float64(11779628.87))

This cell performs an inner join between daily sales and daily exchange rates, which drops dates lacking a rate (such as weekends and holidays) and loses significant sales data without raising an error.


In [27]:
check = sales.merge(rates, on="date", how="left", validate="one_to_one", indicator=True)
lost = check[check["_merge"] == "left_only"]
len(lost), (lost["date"].dt.dayofweek >= 5).sum()


(189, np.int64(172))

This cell uses a left join with an indicator flag to analyze the missing dates, revealing that most lost days are weekends (Saturdays and Sundays) and holidays.


In [28]:
dollars = pd.merge_asof(sales, rates[["date", "sell"]], on="date", direction="backward")
dollars["sales_usd"] = dollars["sales_brl"] / dollars["sell"]
len(dollars), dollars["sell"].isna().sum(), round(dollars["sales_usd"].sum(), 2)

(602, np.int64(0), np.float64(4670445.6))

This cell uses `merge_asof` with a backward direction to assign the most recent available exchange rate to every day (including weekends and holidays), then converts daily sales from BRL to USD.


In [29]:
monthly = (
    dollars.assign(month=dollars["date"].dt.to_period("M"))
    .groupby("month")[["sales_brl", "sales_usd"]]
    .sum()
    .round(2)
)
monthly

,sales_brl,sales_usd
month,,
2017-01,136943.46,43167.83
2017-02,283561.69,91227.88
2017-03,425617.96,136029.98
2017-04,405848.61,129210.32
2017-05,582710.83,181539.13
2017-06,499652.24,151889.87
2017-07,578753.73,180565.46
2017-08,661903.52,210022.19
2017-09,717102.72,228957.83


This cell aggregates the daily sales data into monthly totals for both Brazilian Reais and US Dollars.

In [30]:
august = monthly.loc[[pd.Period("2017-08"), pd.Period("2018-08")]]
(august.iloc[1] / august.iloc[0] - 1).mul(100).round(1)


sales_brl    50.6
sales_usd    23.5
dtype: float64

This cell calculates the year-over-year growth rate from August 2017 to August 2018 for both currencies to compare local vs. dollarized performance.

In [31]:
#5.a
max_brl_month = monthly["sales_brl"].idxmax()
max_usd_month = monthly["sales_usd"].idxmax()
max_brl_month, max_usd_month

(Period('2017-11', 'M'), Period('2017-11', 'M'))

This cell finds the specific months that generated the highest total sales in Brazilian Reais and US Dollars respectively.

In [32]:
#5.b
dollars_buy = pd.merge_asof(sales, rates[["date", "buy"]], on="date", direction="backward")
dollars_buy["sales_usd_buy"] = dollars_buy["sales_brl"] / dollars_buy["buy"]
total_usd_sell = dollars["sales_usd"].sum()
total_usd_buy = dollars_buy["sales_usd_buy"].sum()
total_usd_buy, total_usd_buy - total_usd_sell

(np.float64(4671297.366538351), np.float64(851.7615654040128))

This cell recalculates total dollar sales using the buying rate instead of the selling rate to evaluate the financial difference.

In [33]:
#5.c
p17 = monthly.loc["2017-01":"2017-08"][["sales_brl", "sales_usd"]].sum()
p18 = monthly.loc["2018-01":"2018-08"][["sales_brl", "sales_usd"]].sum()
(p18 / p17 - 1).mul(100).round(1)

sales_brl    140.4
sales_usd    118.5
dtype: float64

This cell compares the cumulative sales from January to August between 2017 and 2018 for both BRL and USD currencies.
